Notebook: Análise de Dados Climáticos de 2024
1. Importar as bibliotecas
As bibliotecas usadas foram pandas e altair

In [10]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

2. Carregar os dados
Os dados foram obtidos a partir de arquivos CSV disponibilizados pelo INMET, sendo referentes ao ano de 2024 para possibilitar uma análise detalahada de todos os fatores que influenciaram as enchentes naquele ano.

In [11]:
# Observação: CSV com separador ';' e vírgula como decimal
file_path = "dados_climaticos_belem_novo.csv"

# Ler CSV
df = pd.read_csv(file_path, sep=';', decimal=',', encoding='latin1', skiprows=8, header=0)

# Visualizar primeiras linhas
print(df.head())
print(df.shape)

         Data  Hora UTC  PRECIPITAÇÃO TOTAL, HORÁRIO (mm)  \
0  2024/01/01  0000 UTC                               0.0   
1  2024/01/01  0100 UTC                               0.0   
2  2024/01/01  0200 UTC                               0.0   
3  2024/01/01  0300 UTC                               0.0   
4  2024/01/01  0400 UTC                               0.0   

   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  \
0                                             1015.9       
1                                             1016.5       
2                                             1016.4       
3                                             1015.8       
4                                             1015.4       

   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)  \
0                                           1015.9   
1                                           1016.5   
2                                           1016.6   
3                                           1016.4   
4 

3. Limpeza e preparação dos dados
Para este conjunto de dados, foram renomeados os atributos, para facilitar sua visualização, além da data e hora que foram convertidas em datetime. Fora isso, para auxiliar na compreenão dos gráficos, foi utilizada uma medida de integração ao empregar a média entre todos os valores obtidos em determinado dia. 

In [12]:
# Renomear colunas para facilitar uso
df.columns = [
    "data", "hora", "precipitacao", "pressao",
    "pressao_max", "pressao_min", "radiacao",
    "temperatura", "ponto_orvalho", "temp_max",
    "temp_min", "orvalho_max", "orvalho_min",
    "umidade_max", "umidade_min", "umidade",
    "vento_direcao", "vento_rajada", "vento_velocidade", "extra"
]

# Converter data + hora em datetime
df['hora'] = df['hora'].str.replace(' UTC', '', regex=False)
df['datetime'] = pd.to_datetime(df['data'] + ' ' + df['hora'], format='%Y/%m/%d %H%M')

# Remover colunas antigas se quiser
df = df.drop(columns=['data', 'hora'])

# Remover valores nulos
df = df.dropna(how='all')

# Conferir estrutura
print(df.info())

# =========================================
# 4. Análise exploratória simples
# =========================================

print(df.describe())

# =========================================
# 3. MÉDIAS DIÁRIAS
# =========================================

df_daily = df.resample('D', on='datetime').mean().reset_index()

# Criar coluna de mês (para filtro)
df_daily['mes'] = df_daily['datetime'].dt.month

<class 'pandas.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   precipitacao      8784 non-null   float64       
 1   pressao           8784 non-null   float64       
 2   pressao_max       8784 non-null   float64       
 3   pressao_min       8784 non-null   float64       
 4   radiacao          8784 non-null   float64       
 5   temperatura       8784 non-null   float64       
 6   ponto_orvalho     8784 non-null   float64       
 7   temp_max          8784 non-null   float64       
 8   temp_min          8784 non-null   float64       
 9   orvalho_max       8784 non-null   float64       
 10  orvalho_min       8784 non-null   float64       
 11  umidade_max       8784 non-null   int64         
 12  umidade_min       8784 non-null   int64         
 13  umidade           8784 non-null   int64         
 14  vento_direcao     8784 non-null   i

4. Visualizações com Altair
Possibilitamos ao usuário escolher entre os meses de 2024, para comparar as diferentes medidas obtidas em cada um, bem como seus gráficos.

In [13]:
# =========================================
# 4. FILTRO INTERATIVO POR MÊS
# =========================================
mes_selecao = alt.selection_point(fields=['mes'], bind=alt.binding_range(min=1, max=12, step=1, name="Mês"), value=1)

# =========================================
# 5. Visualizações com Altair
# =========================================

# --------- 🌡️ Temperatura ao longo do tempo ---------
chart_temp = alt.Chart(df).mark_line().encode(
    x='datetime:T',
    y='temperatura:Q',
    tooltip=['datetime', 'temperatura']
).properties(
    title='Temperatura ao longo do tempo'
).interactive()

chart_temp

alt.Chart(...)

Precipitação

In [14]:
chart_precip = alt.Chart(df_daily).mark_bar(color='purple').encode(
    x='datetime:T',
    y='precipitacao:Q',
    tooltip=['datetime', 'precipitacao']
).add_params(
    mes_selecao
).transform_filter(
    mes_selecao
).properties(
    title='Precipitação diária'
).interactive()
chart_precip

alt.Chart(...)

Umidade

In [15]:
chart_umidade = alt.Chart(df_daily).mark_line(color='green').encode(
    x='datetime:T',
    y='umidade:Q',
    tooltip=['datetime', 'umidade']
).properties(
    title='Umidade relativa do ar'
).interactive()

chart_umidade

alt.Chart(...)

🌡️ Comparação temperatura min/max

In [16]:
chart_temp_range = alt.Chart(df_daily).transform_fold(
    ['temp_min', 'temp_max'],
    as_=['tipo', 'valor']
).mark_line().encode(
    x='datetime:T',
    y='valor:Q',
    color='tipo:N'
).properties(
    title='Temperatura mínima vs máxima'
).interactive()

chart_temp_range

# =========================================
# 6. Possíveis melhorias
# =========================================

# - Filtrar por período - sim
# - Criar médias diárias - com certeza
# - Criar dashboard com múltiplos gráficos

alt.Chart(...)

Temperatura média diária

In [17]:
chart_temp = alt.Chart(df_daily).mark_line().encode(
    x='datetime:T',
    y='temperatura:Q',
    tooltip=['datetime', 'temperatura']
).add_params(
    mes_selecao
).transform_filter(
    mes_selecao
).properties(
    title='Temperatura média diária'
).interactive()

chart_temp

alt.Chart(...)

🔥 Dias mais quentes e frios

In [ ]:
chart_extremos = alt.Chart(df_daily).mark_point(size=60).encode(
    x='datetime:T',
    y='temperatura:Q',
    color=alt.condition(
        alt.datum.temperatura > df_daily['temperatura'].mean(),
        alt.value('red'),
        alt.value('blue')
    ),
    tooltip=['datetime', 'temperatura']
).add_params(
    mes_selecao
).transform_filter(
    mes_selecao
).properties(
    title='Dias mais quentes (vermelho) vs frios (azul)'
)

chart_extremos

alt.Chart(...)

💧 Correlação temperatura vs umidade

In [ ]:
chart_corr_temp_umidade = alt.Chart(df_daily).mark_circle(size=60).encode(
    x='temperatura:Q',
    y='umidade:Q',
    tooltip=['temperatura', 'umidade']
).properties(
    title='Correlação: Temperatura vs Umidade'
)
chart_corr_temp_umidade

alt.Chart(...)

💨 Correlação vento (rajada) vs precipitação

In [ ]:
chart_corr_vento = alt.Chart(df_daily).mark_circle(size=60, color='orange').encode(
    x='vento_rajada:Q',
    y='precipitacao:Q',
    tooltip=['vento_rajada', 'precipitacao']
).properties(
    title='Correlação: Rajada de vento vs Precipitação'
)
chart_corr_vento

alt.Chart(...)